In [ ]:
import glob, os, shutil, time, json
T0 = time.time()
def log(*a):
    print(f'[{time.time() - T0:7.1f}s]', *a, flush=True)
R = {}
shards = sorted(glob.glob('/kaggle/input/**/Qwen-Image-Layered/text_encoder/*.safetensors', recursive=True))
gguf = glob.glob('/kaggle/input/**/qwen-image-layered-Q4_K_S.gguf', recursive=True)
log('shards', [(os.path.basename(s), round(os.path.getsize(s) / 1e9, 2)) for s in shards])
log('gguf', gguf)
os.system('df -h /tmp /kaggle/working | tail -n 2; mount | grep -i kaggle/input | head -3')
shard = min(shards, key=os.path.getsize)          # smallest shard (~1.7 GB) keeps the test short
size_gb = os.path.getsize(shard) / 1e9

# B: direct safetensors reads from the attached input (as transformers does), 40 tensors max, 5 min cap
from safetensors import safe_open
t = time.time(); n = 0; nbytes = 0
with safe_open(shard, 'pt') as f:
    for k in list(f.keys()):
        x = f.get_tensor(k); n += 1; nbytes += x.numel() * x.element_size()
        if n >= 40 or time.time() - t > 300:
            break
R['B_direct_tensors'] = n; R['B_direct_s'] = round(time.time() - t, 1); R['B_direct_MBps'] = round(nbytes / 1e6 / max(time.time() - t, 1e-3), 1)
log('B direct read:', R['B_direct_tensors'], 'tensors in', R['B_direct_s'], 's =', R['B_direct_MBps'], 'MB/s')

# C: sequential copy to local disk, then load everything from the local copy
t = time.time(); dst = '/tmp/' + os.path.basename(shard); shutil.copyfile(shard, dst)
R['C_copy_s'] = round(time.time() - t, 1); R['C_copy_MBps'] = round(size_gb * 1e3 / max(time.time() - t, 1e-3), 1)
log('C copy', round(size_gb, 2), 'GB in', R['C_copy_s'], 's =', R['C_copy_MBps'], 'MB/s')
t = time.time(); n = 0
with safe_open(dst, 'pt') as f:
    for k in f.keys():
        f.get_tensor(k); n += 1
R['C_local_load_tensors'] = n; R['C_local_load_s'] = round(time.time() - t, 1)
log('C local load: all', n, 'tensors in', R['C_local_load_s'], 's')
os.remove(dst)

# D: sequential read throughput of the GGUF (first 3 GB)
if gguf:
    t = time.time(); read = 0
    with open(gguf[0], 'rb', buffering=0) as fh:
        while read < 3e9:
            b = fh.read(64 * 1024 * 1024)
            if not b:
                break
            read += len(b)
    R['D_gguf_seq_MBps'] = round(read / 1e6 / max(time.time() - t, 1e-3), 1)
    log('D gguf sequential read:', round(read / 1e9, 2), 'GB at', R['D_gguf_seq_MBps'], 'MB/s')
R['total_s'] = round(time.time() - T0, 1)
log(json.dumps(R))
json.dump(R, open('/kaggle/working/cache_speed.json', 'w'), indent=1)